# 싼타페 설명서 데이터 분석 [프로젝트 추가]
저장된 analysis_results/summary.json을 읽어 데이터 분포·기술 통계·점검 결과를 재현합니다. DB 조회·API·임베딩 호출은 없습니다. 스냅샷을 갱신하려면 개인 data_analysis 모듈을 별도로 실행합니다. 단일 설명서 분석이며 가설 검정·답변 정확도 평가는 아닙니다. 기존 노트북 셀은 변경하지 않았습니다.

In [ ]:
# 현재 작업 위치에서 프로젝트 루트를 찾고 집계 결과만 읽습니다.
import json
from pathlib import Path
import pandas as pd
import altair as alt
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/car_search_rag/zzong_santafe_lag').is_dir())
folder = root / 'src/car_search_rag/zzong_santafe_lag/analysis_results'
summary = json.loads((folder / 'summary.json').read_text(encoding='utf-8'))
frame = pd.DataFrame(summary['parents'])
print(summary['created_at'], summary['counts'])
pd.DataFrame(summary['length_statistics'])

In [ ]:
# 유형별 빈도로 어떤 자료가 검색 대상인지 살펴봅니다. 검토 비율은 정답률이 아닙니다.
types = pd.DataFrame([{'type': k, 'count': v} for k, v in summary['type_counts'].items()])
alt.Chart(types).mark_bar().encode(x='count:Q', y='type:N', tooltip=['type', 'count'])

In [ ]:
# 본문 길이의 분포를 확인합니다. 문자 수는 토큰 수와 다릅니다.
alt.Chart(frame).mark_bar().encode(x=alt.X('characters:Q', bin=alt.Bin(maxbins=20)), y='count():Q')

In [ ]:
# 순위 상관은 길이와 청크 수의 관계를 요약할 뿐 검색 품질·인과관계를 입증하지 않습니다.
print('순위 상관:', summary['length_chunk_rank_correlation'])
alt.Chart(frame).mark_circle().encode(x='characters:Q', y='chunks:Q', tooltip=['title', 'characters', 'chunks'])

In [ ]:
# 구조 점검에서 0건이어도 본문의 의미·수치가 모두 검토됐다는 뜻은 아닙니다.
print(summary['limitations'])
pd.DataFrame([{'check': k, 'count': v} for k, v in summary['quality_checks'].items()])

## 50문항 평가 결과 표시 [프로젝트 추가]
평가 결과는 저장 파일에서만 읽습니다. 통과 29·부분 3·실패 18은 초기 응답에 대한 AI 잠정 판정이며, 사용자 확인·생성 답변·Hit@5·MRR 평가는 남아 있습니다. 대시보드의 유형별 판정과 문항 목록을 재현하는 추가 셀입니다. 기존 셀·저장 결과는 보존했습니다.

In [ ]:
# 질문을 재실행하지 않고 같은 평가 기록을 읽습니다. 사용자 확정 판정으로 취급하지 않습니다.
evaluation_folder = folder.parent / 'evaluation_50_results'
evaluation_summary = json.loads((evaluation_folder / 'summary.json').read_text(encoding='utf-8'))
evaluation_frame = pd.DataFrame(json.loads((evaluation_folder / 'graded_results.json').read_text(encoding='utf-8')))
print(evaluation_summary['ai_provisional_counts'], '사용자 확정:', evaluation_summary['human_confirmed'])
evaluation_frame[['case_id', 'question', 'overall', 'source_grade', 'response_grade', 'note']]

## 우선 미검토 자료 8개 원본 대조 [프로젝트 추가]
2026-10-06 source_reviews/20261006_batch2/report.md: 관련 원본 및 경계 38쪽 대조, 본문 보완안 7개·본문 변경 불필요 1개, 그림 연결 26개 중 다른 항목 3개 제외안. 원본 raw_text 보존. 개별 그림 파일 검증·필수 문맥·새 벡터 버전 준비는 남아 있습니다. 사용자 직접 평가 종료 후 반영하기로 했으므로 운영 상태는 변경하지 않았습니다. 전체 미검토 502개 중 이번 대조 외 494개는 남아 있습니다.

## [프로젝트 추가] 사용자 실행 캡처 50문항 통합 기록 (2026-10-06)

- 사용자 실행 화면을 Codex가 판정한 기록과 자동 초기 응답 평가를 별도 파일로 보존합니다.
- 직접 화면: 통과 21 / 부분 2 / 실패 27. 자동 초기 응답: 통과 29 / 부분 3 / 실패 18.
- 직접 실행에서 생성 답변을 확인한 문항은 16개입니다. 사용자 최종 판정 확정과 Hit@5·MRR 산출은 하지 않았습니다.
- B04 추가 캡처, C07·C08 번호 정정, C18 입력 질문 미표시를 기록에 반영했습니다.
- 두 평가는 단계와 판정 기준이 달라 성능 개선·하락률로 비교하지 않습니다. 원문 후보를 찾았지만 답변을 보류한 경우 검색과 화면 대응을 나눴습니다.
- B03·B04 핵심 값 통과 판정은 유지하되 일반 F선 주의사항의 장치별 적용 범위는 미확인으로 기록했습니다.
- 파일: evaluation_50_results/screenshot_evaluation.json, screenshot_evaluation.csv, output/pdf/santafe_evaluation_combined.pdf.
- 싼타페 개인 대시보드에 비교 목록·응답 요약·보완 메모·다운로드를 추가했습니다. 기존 자동 결과 및 원문 DB의 검토 상태는 변경하지 않았습니다.

## [프로젝트 추가] 상위 주제·구체 질문·복합 맥락 추가 30문항 (2026-10-06)

- A/B/C 각 10문항. 같은 10개 기능 영역을 질문 구체성과 의미 맥락에 따라 변형했습니다. C마다 3개 의미 맥락을 기록합니다.
- 질문·기대 근거를 실행 전에 고정하고 기존 검색/검토/생성 버전으로 독립 질문 30개를 실행했습니다. 서비스에 정답을 전달하지 않았습니다.
- 검색 30회, 생성 동작 요청 14회. 생성 답변 5개, 생성 검증 후 원문 발췌 복귀 9개, 실행 예외 0건. 생성 검증 복귀는 정상 실행과 별도로 표시합니다.
- Codex 잠정 최종 응답 판정: A 통과 8/부분 1/실패 1, B 통과 1/실패 9, C 실패 10. A는 주제 관련 위치 안내·원문 선택지를 허용하고 B/C는 목적에 직접 답해야 하므로 단순 유형 간 정답률 비교에 한계가 있습니다.
- 상위 5개에 관련 근거가 있었던 문항은 29/30입니다. 후보 관련성은 원본 대조를 통한 Codex 잠정 판정이며 사용자 확정 골든셋 점수와 구분합니다.
- 검색에 정답이 있어도 검토된 무관한 자료를 선택하는 문제, 미검토 답변 보류, 생성 검증 실패 시 무관한 원문 복귀를 분리해 기록합니다.
- 질문 목록·보고서·CSV·원시 결과·고정 계획은 evaluation_30_results에 저장합니다. 기존 50문항 파일 및 DB 검토 상태는 유지합니다.
- 개인 대시보드에 추가 30문항 평가 탭을 추가했습니다. 화면을 열 때 재검색하거나 생성 API를 호출하지 않습니다.
- [프로젝트 추가] 2026-10-07: 사용자 승인으로 공통 앱의 싼타페 메뉴에도 자료 화면을 연결했습니다. 화면 전환·복귀·대화 보존·차종 분리 테스트 3개 통과. 기존 팀 테스트 32개 중 26개 통과, 가짜 설정/응답 누락 오류 6개는 남아 있습니다. 실제 원격 확인은 배포 화면에서 별도로 수행합니다.

## 원문 8개 반영과 재평가 2026-10-07 [프로젝트 추가]

사용자 승인 범위 8개를 개인 DB의 별도 검토 버전으로 반영했습니다. 기존 원문과 5개 검토 자료의 벡터를 보존했습니다. 새 청크 68개, 누적 버전 96개. 그림 26개 바이트 대조 후 23개 설명 보완·3개 잘못된 연결 제외.

현재 527개 부모 중 검토 33개·미검토 494개입니다. 기존 질문 16개의 기대 출처 선택은 0/16에서 10/16으로 변했습니다. 기존 자료 회귀 3/3 유지, 실제 DB 대표 3문항 검색·생성 확인. 출처 선택과 생성 성공은 정답률이 아니며 남은 문제와 원문 검토 범위를 보고서에 기록했습니다.

source_reviews/20261007_batch2/report.md 및 review_summary.json을 대시보드와 작업 Document에서 읽습니다. 기존 평가 결과와 노트북 셀·출력은 보존합니다. 검토 주체는 Codex이며 사용자 확정 골든셋으로 표시하지 않습니다.
